# MiniMax H3 影片工作站（手動操作 + agent 送件）

右上角 ▼ →「變更執行階段類型」→ **A100** → 儲存 → 依序執行 **① 準備環境**；需要讓 agent 自動送件時，再執行 **② 代理模式**。

> ⚠️ 這份筆記本**不開公網、不建隧道**（Colab 條款禁止遠端代理與對外服務）。agent 只透過你 Google Drive 裡的 `h3-queue` 資料夾送件、取件。


In [ ]:
#@title ① 準備環境並開啟操作畫面（首次約數分鐘～20 分鐘）
model_precision = "INT8＋INT8 文字編碼器（建議）" #@param ["INT8＋INT8 文字編碼器（建議）", "INT8（NVFP4 文字編碼器，較省空間）", "未量化（A100 80GB）"]
download_omni = True #@param {type:"boolean"}
download_ref4 = True #@param {type:"boolean"}
#@markdown 勾選 download_ref4＝另外下載 Ref2VA INT8 與 4 步 LoRA（約 +23GB），才能用 ref4 草稿模式與 ref2va／ref2va_hd／ayase（ayase 另含 HyperFlow LoRA、潛空間放大節點與模型，約 +2GB）。
#@markdown 勾選＝另外下載 FastH3 Omni（約 +22GB，INT8），才能用 Omni 參考圖模式。
import sys, types, hashlib
BACKEND_SOURCE = '"""Self-contained MiniMax H3 backend for a Colab notebook. No public server."""\nfrom __future__ import annotations\n\nimport base64\nimport html\nimport io\nimport json\nimport os\nimport secrets\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport uuid\nimport zipfile\nfrom pathlib import Path\n\nBASE = Path(os.environ.get("H3_LAB_DIR", "/content/h3-lab"))\nCOMFY = BASE / "ComfyUI"\nOUTPUT = BASE / "results"\nPORT = 8193\nURL = f"http://127.0.0.1:{PORT}"\n# ce42de5f (2026-10-18) adds "models with PDD baked in", which FastH3 Omni needs; the older pin fails in sampling.\nCOMFY_COMMIT = "ce42de5f64ac1d421e304ab7b6c0b3d2f0dde1f6"\nVHS_COMMIT = "4d907bee61e92c2e65af3bd6383a4e4d356126d1"\nMODEL_REVISION = "e5eb578a89295337b8ff433a035929ce0279e0b6"\nCOMMON = {"audio": "vae/minimax_h3_audio_vae_fp32.safetensors", "lora": "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", "lora4": "loras/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"}\nPROFILES = {\n    "int8": {"label": "INT8（量化）", "min_vram": 38, "min_ram": 60, "free_disk": 58,\n             "files": {**COMMON, "model": "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "text": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "video": "vae/minimax_h3_video_vae_int8_convrot.safetensors"}},\n    "full": {"label": "BF16 / FP16（非量化）", "min_vram": 75, "min_ram": 120, "free_disk": 120,\n             "files": {**COMMON, "model": "diffusion_models/minimax_h3_fl2va_pruned_bf16.safetensors", "text": "text_encoders/qwen3vl_32b_minimax_h3_bf16.safetensors", "video": "vae/minimax_h3_video_vae_fp16.safetensors"}},\n}\nOMNI_REPO = "FastVideo/FastH3-Omni-Comfy"\nOMNI_REVISION = "82eff2a6eee6c962846879f4bab52610a4df24d3"\nOMNI_FILES = {"int8": "diffusion_models/fasth3_omniref_8step_pruned_int8_convrot.safetensors", "full": "diffusion_models/fasth3_omniref_8step_pruned_bf16.safetensors"}\nOMNI_FILES["int8enc"] = OMNI_FILES["int8"]\nOMNI_DISK_GIB = {"int8": 22, "int8enc": 22, "full": 42}\nMAX_REFS = 9\nIMAGE_NAMES = ("first_frame", "last_frame") + tuple(f"ref_{i}" for i in range(1, MAX_REFS + 1))\nSAMPLERS = {"res_multistep+simple": ("res_multistep", "simple"), "euler+beta": ("euler", "beta"), "euler+simple": ("euler", "simple")}\nPROFILES["int8enc"] = {**PROFILES["int8"], "label": "INT8（INT8 文字編碼器）", "free_disk": 70,\n                       "files": {**PROFILES["int8"]["files"], "text": "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"}}\n# Ref2VA INT8 + ref2v turbo 4-step LoRA: the alternative draft engine to compare against Omni.\nREF4_FILES = {"model": "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors", "lora": "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"}\nREF4_DISK_GIB = 23\n# Ayase Studio recipe: HyperFlow 8-step LoRA, 8-step draft at ~0.5 MP, neural latent upscale, low-denoise second pass.\nAYASE_LORA = ("drbaph/MiniMax-H3-Turbo-Lora-ComfyUI", "bb2bc497cbaca89dadd0bcf1856eed4f8275be20", "minimax_h3_hyperflow_8step_v1.0_comfyui_pruned_bf16_resized_avg_rank_20_bf16.safetensors")\nAYASE_UPSCALER = ("LBH-123-AI/Minimax_h3_latent_Upscaler", "3f941d5d182014dd5c0a5e16330420ee2d4aa0c6", "minimax_h3_latent_upscaler_3d_conv_v1/minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors")\nAYASE_UPSCALER_NAME = "minimax_h3_latent_upscaler_3d_fp16.safetensors"  # the node only lists files whose name has "3d" and "fp16"\nAYASE_NODES = ("https://github.com/LBH-123-AI/Comfyui_Minimax_h3_latent_Upscaler.git", "40316cf008b2fd8663263270669eb4da23f89d2c")  # MIT, user-approved, pinned\nAYASE_DISK_GIB = 2\nAYASE_REPAIR = "修復一採中的結構畸形、重影、重複或缺失部件、破損邊緣、紋理斷裂和閃爍；嚴格保持原劇情時序、運鏡、構圖、主體身份、服裝、動作、場景、畫面風格和正常區域不變。"\nREF_MODES = ("omni", "omni_hd", "ref4", "ref2va", "ref2va_hd", "ayase")\nHD_MODES = ("omni_hd", "ref2va_hd")\n# draft size -> (render w, render h, crop w, crop h, crop x, crop y). 720 is not on H3\'s 32-px grid, so render 736 and crop in-graph.\nHD_TARGETS = {"p480": (1280, 736, 1280, 720, 0, 8), "p480v": (736, 1280, 720, 1280, 8, 0)}\nSIZES = {"p480": (832, 480), "p480v": (480, 832), "landscape": (960, 544), "portrait": (544, 960), "square": (768, 768), "native": (1344, 768), "p720": (1280, 704)}\nFRAMES = {"5": 124, "8": 192, "10": 243, "15": 362}\n_jobs: dict[str, dict] = {}\n_lock = threading.Lock()\n_active: str | None = None\n_process = None\n\n\ndef _request(method, route, **kwargs):\n    import requests\n    response = requests.request(method, URL + route, timeout=kwargs.pop("timeout", 30), **kwargs)\n    if not response.ok:\n        raise RuntimeError(f"ComfyUI {response.status_code}: {response.text[:2500]}")\n    return response.json()\n\n\ndef _run(args, **kwargs):\n    if "stdout" in kwargs:\n        subprocess.run(args, check=True, **kwargs)\n    else:\n        with (BASE / "setup.log").open("a") as log:\n            subprocess.run(args, check=True, stdout=log, stderr=subprocess.STDOUT, **kwargs)\n\n\ndef _checkout(url, destination, commit):\n    if not (destination / ".git").exists():\n        destination.mkdir(parents=True, exist_ok=True)\n        _run(["git", "init", str(destination)])\n        _run(["git", "-C", str(destination), "remote", "add", "origin", url])\n    current = subprocess.run(["git", "-C", str(destination), "rev-parse", "HEAD"], capture_output=True, text=True)\n    if current.returncode or current.stdout.strip() != commit:\n        _run(["git", "-C", str(destination), "fetch", "--depth", "1", "origin", commit])\n        _run(["git", "-C", str(destination), "checkout", "--detach", commit])\n\n\ndef hardware():\n    """Works before pip/torch setup, including CPU-only runtimes."""\n    info = dict(gpu=None, vram_gib=0, ram_gib=0, free_disk_gib=0)\n    try:\n        result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=15)\n        if result.returncode == 0 and result.stdout.strip():\n            name, memory = result.stdout.strip().splitlines()[0].rsplit(",", 1)\n            info.update(gpu=name.strip(), vram_gib=float(memory.strip()) / 1024)\n    except (OSError, ValueError, subprocess.TimeoutExpired):\n        pass\n    try:\n        info["ram_gib"] = os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 2**30\n    except (ValueError, OSError):\n        import psutil\n        info["ram_gib"] = psutil.virtual_memory().total / 2**30\n    info["free_disk_gib"] = shutil.disk_usage(BASE if BASE.exists() else Path.cwd()).free / 2**30\n    return info\n\n\ndef gpu_issues(info, profile="int8"):\n    selected = PROFILES[profile]\n    if not info["gpu"]:\n        return ["未偵測到 GPU"]\n    issues = []\n    if info["vram_gib"] < selected["min_vram"]:\n        issues.append("GPU 記憶體不足")\n    if info["ram_gib"] < selected["min_ram"]:\n        issues.append("主記憶體不足")\n    return issues\n\n\ndef gpu_card(info, profile="int8"):\n    issues = gpu_issues(info, profile)\n    label = " / ".join(issues) if issues else "GPU OK"\n    color = "#9b620a" if issues else "#14765e"\n    device = html.escape(info["gpu"] or "無 GPU")\n    instructions = []\n    if not info["gpu"] or info["vram_gib"] < PROFILES[profile]["min_vram"]:\n        instructions.append("選擇「A100 GPU」")\n    if info["gpu"] and info["ram_gib"] < PROFILES[profile]["min_ram"]:\n        instructions.append("開啟「高 RAM」")\n    guide = (\'<ol style="padding-left:22px;margin:10px 0 0">\'\n             \'<li>右上角「連線」旁的 ▼ →「變更執行階段類型」</li>\'\n             + \'\'.join(\'<li>\' + step + \'</li>\' for step in instructions)\n             + \'<li>「儲存」並重新連線後，再按一次 ▶</li></ol>\') if issues else ""\n    if not info["gpu"] or info["vram_gib"] < PROFILES[profile]["min_vram"]:\n        guide += \'<p style="margin:8px 0 0;font-size:12px">若無法選擇 A100，請確認 Colab 的方案與剩餘用量。不支援免費的 T4。</p>\'\n    if issues and profile == "full":\n        guide += \'<p>非量化版需要 A100 80GB。也可以在模型設定中選擇「INT8」。</p>\'\n    return f"""<div style="font:14px/1.65 system-ui;padding:16px;border:1px solid #dce3ef;border-radius:12px;max-width:700px">\n    <b style="color:{color}">{html.escape(label)}</b><div>{device} · GPU {info[\'vram_gib\']:.0f} GB · RAM {info[\'ram_gib\']:.0f} GB</div>{guide}\n    <details style="margin-top:10px;font-size:12px"><summary>檢查指令</summary><code>!nvidia-smi</code></details></div>"""\n\n\nclass SetupProgress:\n    """One output card, updated even while a download or pip command runs."""\n    def __init__(self):\n        self.started = time.monotonic()\n        self.message = "檢查 GPU 中"\n        self.step = 0\n        self.error = False\n        self.stopped = threading.Event()\n        from IPython.display import HTML, display\n        self.handle = display(HTML(self.render()), display_id=True)\n        self.thread = threading.Thread(target=self.heartbeat, daemon=True)\n        self.thread.start()\n\n    def render(self):\n        elapsed = int(time.monotonic() - self.started)\n        color = "#922c36" if self.error else "#4c48d9"\n        return f\'<div style="font:14px/1.6 system-ui;max-width:700px;padding:16px;border:1px solid #dce3ef;border-radius:12px"><b style="color:{color}">{html.escape(self.message)}</b><div style="color:#68758d;font-size:12px">已耗時 {elapsed//60} 分 {elapsed%60:02d} 秒 · 首次預估 數分鐘至 20 分鐘（依網路而異）</div><progress aria-label="準備流程" max="3" value="{self.step}" style="width:100%;accent-color:{color}"></progress></div>\'\n\n    def update(self, message, step):\n        from IPython.display import HTML\n        self.message, self.step = message, step\n        self.handle.update(HTML(self.render()))\n\n    def heartbeat(self):\n        from IPython.display import HTML\n        while not self.stopped.wait(1):\n            self.handle.update(HTML(self.render()))\n\n    def close(self):\n        self.stopped.set()\n        self.thread.join(timeout=2)\n\n\ndef setup(profile="int8", omni=True, ref4=False):\n    if _active:\n        record = json.loads((BASE / "setup.json").read_text())\n        if record.get("profile") == profile:\n            return record  # Reopen the UI and reconnect to the running job.\n        raise RuntimeError("影片生成中無法變更設定。請等待完成或中止後再執行。")\n    if profile not in PROFILES:\n        raise ValueError("模型設定不正確。")\n    BASE.mkdir(parents=True, exist_ok=True)\n    OUTPUT.mkdir(exist_ok=True)\n    selected = PROFILES[profile]\n    info = hardware()\n    if gpu_issues(info, profile):\n        from IPython.display import HTML, display\n        display(HTML(gpu_card(info, profile)))\n        return None\n    if ref4 and profile == "full":\n        raise RuntimeError("Ref2VA 4 步草稿目前只支援 INT8 設定。")\n    wanted = list(selected["files"].values()) + ([OMNI_FILES[profile]] if omni else []) + (list(REF4_FILES.values()) if ref4 else []) + (["loras/" + AYASE_LORA[2], "latent_upscale_models/" + AYASE_UPSCALER_NAME] if ref4 else [])\n    need_disk = selected["free_disk"] + (OMNI_DISK_GIB[profile] if omni else 0) + (REF4_DISK_GIB + AYASE_DISK_GIB if ref4 else 0)\n    if info["free_disk_gib"] < need_disk and not all((COMFY / "models" / f).exists() for f in wanted):\n        raise RuntimeError(f"首次執行需要至少 {need_disk} GiB 的可用空間。")\n    progress = SetupProgress()\n    try:\n        if (BASE / "setup.json").exists():\n            cached = json.loads((BASE / "setup.json").read_text())\n            if (cached.get("profile"), cached.get("comfy_commit"), cached.get("vhs_commit"), cached.get("model_revision")) == (profile, COMFY_COMMIT, VHS_COMMIT, MODEL_REVISION) and bool(cached.get("omni")) == omni and bool(cached.get("ref4")) == ref4 and all((COMFY / "models" / f).exists() for f in wanted):\n                progress.update("連線到生成畫面中", 2)\n                start_server(omni)\n                progress.update("準備完成 ↓ 請在下方畫面製作影片", 3)\n                return cached\n        progress.update("1/3 準備環境中", 0)\n        torch_probe = subprocess.run([sys.executable, "-c", "import torch; print(torch.version.cuda or \'0\')"], capture_output=True, text=True, timeout=60)\n        cuda = torch_probe.stdout.strip().split(".")[0] if torch_probe.returncode == 0 else "0"\n        if profile != "full" and int(cuda or "0") < 13:\n            _run([sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0", "torchvision==0.26.0", "torchaudio==2.11.0", "--index-url", "https://download.pytorch.org/whl/cu130"])\n        if (COMFY / ".git").exists():\n            head = subprocess.run(["git", "-C", str(COMFY), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()\n            if head != COMFY_COMMIT:  # a server started from the old checkout would keep running old code\n                subprocess.run(["pkill", "-f", str(COMFY / "main.py")])\n                time.sleep(3)\n        _checkout("https://github.com/Comfy-Org/ComfyUI.git", COMFY, COMFY_COMMIT)\n        vhs = COMFY / "custom_nodes" / "ComfyUI-VideoHelperSuite"\n        _checkout("https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git", vhs, VHS_COMMIT)\n        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt"), "-r", str(vhs / "requirements.txt"), "huggingface_hub", "hf_xet", "psutil", "requests", "Pillow", "imageio-ffmpeg", "websocket-client"])\n        downloads = [("Comfy-Org/MiniMax-H3", name, MODEL_REVISION) for name in selected["files"].values()]\n        if omni:\n            downloads.append((OMNI_REPO, OMNI_FILES[profile], OMNI_REVISION))\n        if ref4:\n            downloads += [("Comfy-Org/MiniMax-H3", name, MODEL_REVISION) for name in REF4_FILES.values()]\n        for index, (repo, name, revision) in enumerate(downloads):\n            progress.update(f"2/3 下載模型中 · {index + 1}/{len(downloads)}", 1 + index / len(downloads))\n            script = "from huggingface_hub import hf_hub_download; import sys; hf_hub_download(sys.argv[1],sys.argv[2],revision=sys.argv[3],local_dir=sys.argv[4])"\n            _run([sys.executable, "-c", script, repo, name, revision, str(COMFY / "models")], env={**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_XET_HIGH_PERFORMANCE": "1"})\n        if ref4:  # Ayase Studio assets ride on the download_ref4 box: HyperFlow LoRA, neural latent upscaler weights and node pack\n            _checkout(AYASE_NODES[0], COMFY / "custom_nodes" / "Comfyui_Minimax_h3_latent_Upscaler", AYASE_NODES[1])\n            script = "from huggingface_hub import hf_hub_download; import sys; hf_hub_download(sys.argv[1],sys.argv[2],revision=sys.argv[3],local_dir=sys.argv[4])"\n            quiet = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_XET_HIGH_PERFORMANCE": "1"}\n            _run([sys.executable, "-c", script, AYASE_LORA[0], AYASE_LORA[2], AYASE_LORA[1], str(COMFY / "models" / "loras")], env=quiet)\n            target = COMFY / "models" / "latent_upscale_models"\n            _run([sys.executable, "-c", script, AYASE_UPSCALER[0], AYASE_UPSCALER[2], AYASE_UPSCALER[1], str(target)], env=quiet)\n            if not (target / AYASE_UPSCALER_NAME).exists():\n                shutil.copy2(target / AYASE_UPSCALER[2], target / AYASE_UPSCALER_NAME)\n        progress.update("3/3 啟動生成引擎中", 2)\n        start_server(omni)\n        record = {"profile": profile, "profile_label": selected["label"], "comfy_commit": COMFY_COMMIT, "vhs_commit": VHS_COMMIT, "model_revision": MODEL_REVISION, "omni": omni, "ref4": ref4, "hardware": hardware(), "setup_seconds": round(time.monotonic() - progress.started, 2)}\n        (BASE / "setup.json").write_text(json.dumps(record, ensure_ascii=False, indent=2))\n        progress.update("準備完成 ↓ 請在下方畫面製作影片", 3)\n        return record\n    except Exception as exc:\n        progress.error = True\n        progress.update("準備已中止，請重新執行準備儲存格", progress.step)\n        from IPython.display import HTML, display\n        log = (BASE / "setup.log").read_text(errors="replace")[-5000:] if (BASE / "setup.log").exists() else ""\n        display(HTML(\'<details><summary>錯誤詳情</summary><pre style="white-space:pre-wrap">\' + html.escape(str(exc) + "\\n" + log) + \'</pre></details>\'))\n        return None\n    finally:\n        progress.close()\n\n\ndef start_server(omni=False):\n    global _process\n    try:\n        _request("GET", "/system_stats", timeout=2)\n        return\n    except Exception:\n        pass\n    log = (BASE / "comfy.log").open("a")\n    # Loopback only: neither a public tunnel nor a public Gradio link is created.\n    _process = subprocess.Popen([sys.executable, str(COMFY / "main.py"), "--listen", "127.0.0.1", "--port", str(PORT), "--disable-auto-launch"], cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)\n    for _ in range(150):\n        if _process.poll() is not None:\n            raise RuntimeError("生成引擎啟動失敗。\\n" + (BASE / "comfy.log").read_text(errors="replace")[-4000:])\n        try:\n            _request("GET", "/system_stats", timeout=2)\n            nodes = _request("GET", "/object_info")\n            for required in ("MiniMaxH3ImageToVideo", "VHS_VideoCombine", "VAEDecodeAudio") + (("MiniMaxH3ReferenceToVideo", "MiniMaxH3SigmaShift") if omni else ()):\n                if required not in nodes:\n                    raise RuntimeError(f"必要節點 {required} 無法載入。請檢查準備紀錄。")\n            return\n        except Exception:\n            time.sleep(2)\n    raise TimeoutError("啟動未能在 5 分鐘內完成。請檢查準備紀錄。")\n\n\ndef clean_config(value, need_refs=True):\n    if not isinstance(value, dict):\n        raise ValueError("設定請以 JSON 物件傳入。")\n    prompt = str(value.get("prompt", "")).strip()\n    if not prompt or len(prompt) > 16000:\n        raise ValueError("請輸入提示詞（16,000 字以內）。")\n    seconds = str(value.get("seconds", "5"))\n    mode = value.get("mode", "turbo")\n    size = value.get("size", "p480" if mode in HD_MODES else "landscape")\n    if size not in SIZES or seconds not in FRAMES or mode not in ("turbo", "turbo4", "omni", "omni_hd", "ref4", "ref2va", "ref2va_hd", "ayase", "quality"):\n        raise ValueError("請確認畫面比例、長度與品質設定。")\n    if mode == "ref2va" and "size" not in value:\n        size = "p720"\n    if mode in HD_MODES and size not in HD_TARGETS:\n        raise ValueError("omni_hd／ref2va_hd 的草稿尺寸只能是 " + " 或 ".join(HD_TARGETS) + "。")\n    refine_steps = int(value.get("refine_steps", 4))\n    if mode == "ref2va_hd" and "refine_steps" not in value:\n        refine_steps = 8  # last 8 of the 20-step schedule\n    if not 2 <= refine_steps <= (12 if mode == "ref2va_hd" else 6):\n        raise ValueError("refine_steps 請設為 2 至 6（ref2va_hd 可到 12）。")\n    steps = {"turbo": 8, "turbo4": 4, "omni": 8, "omni_hd": 8, "ref4": 4, "ayase": 8}.get(mode) or int(value.get("steps", 20))\n    if mode in ("ref2va", "ref2va_hd") and not 12 <= steps <= 30:  # official template default 20; audio needs 12+\n        raise ValueError("ref2va 的步數請設為 12 至 30。")\n    if mode == "ref2va_hd" and refine_steps >= steps:\n        raise ValueError("ref2va_hd 的 refine_steps 必須小於 steps，否則二次取樣會完全重畫、忽略草稿。")\n    if mode == "quality" and steps not in (16, 20, 25, 30):\n        raise ValueError("標準模式的步數請從 16、20、25、30 中選擇。")\n    raw_seed = value.get("seed", -1)\n    seed = int(raw_seed)\n    if isinstance(raw_seed, float) and not raw_seed.is_integer():\n        raise ValueError("Seed 請設為整數。")\n    if seed != -1 and not 0 <= seed <= 2**32 - 1:\n        raise ValueError("Seed 請設為 -1（隨機）或 0 至 4294967295。")\n    if seed == -1:\n        seed = secrets.randbelow(2**32)\n    width, height = SIZES[size]\n    sampler = value.get("sampler", "euler+simple" if mode == "ayase" else "res_multistep+simple")\n    if sampler not in SAMPLERS:\n        raise ValueError("sampler 只能是 " + " 或 ".join(SAMPLERS) + "。")\n    images = {}\n    for key in IMAGE_NAMES:\n        name = str(value.get(key, ""))\n        if name:\n            if Path(name).name != name or not name.startswith("h3ref_") or not (COMFY / "input" / name).is_file():\n                raise ValueError("請重新上傳參考圖片。")\n            images[key] = name\n    if mode in REF_MODES:\n        # Omni takes ordered references (<Picture 1>, <Picture 2>...); the UI first/last slots count as the first two.\n        numbered = [i for i in range(1, MAX_REFS + 1) if f"ref_{i}" in images]\n        if numbered != list(range(1, len(numbered) + 1)):\n            raise ValueError("參考圖片編號必須從 ref_1 開始連續（ref_N 對應提示詞裡的 <Picture N>）。")\n        refs = [images[k] for k in IMAGE_NAMES[2:] if k in images] or [images[k] for k in IMAGE_NAMES[:2] if k in images]\n        if not refs and need_refs:\n            raise ValueError("參考圖模式（omni、omni_hd、ref4、ref2va、ref2va_hd、ayase）至少需要一張參考圖片。")\n        images = {"refs": refs}\n    else:\n        images = {k: v for k, v in images.items() if k in IMAGE_NAMES[:2]}\n    shift = bool(value.get("shift", False))\n    if shift and mode not in ("ref4", "ref2va", "ref2va_hd"):\n        raise ValueError("shift 只能用在 ref4、ref2va、ref2va_hd（Omni 本來就有 shift）。")\n    extra = {}\n    if mode == "ayase":\n        upscale, second_steps, second_denoise = float(value.get("upscale", 2.0)), int(value.get("second_steps", 7)), float(value.get("second_denoise", 0.22))\n        if not 1.25 <= upscale <= 2.5 or not 3 <= second_steps <= 12 or not 0.05 <= second_denoise <= 0.6:\n            raise ValueError("ayase 的 upscale 請設為 1.25 至 2.5、second_steps 3 至 12、second_denoise 0.05 至 0.6。")\n        extra = dict(tiled=bool(value.get("tiled", False)), upscale=upscale, second_steps=second_steps, second_denoise=second_denoise,\n                     target_width=round(width * upscale / 32) * 32, target_height=round(height * upscale / 32) * 32)\n    return dict(**extra, shift=shift, sampler=sampler, refine_steps=refine_steps, prompt=prompt, size=size, seconds=seconds, frames=FRAMES[seconds], actual_seconds=FRAMES[seconds] / 24, mode=mode, steps=steps, seed=seed, width=width, height=height, **images)\n\n\ndef upload_image(data_url, _filename="image"):\n    from PIL import Image, ImageOps\n    if not isinstance(data_url, str) or not data_url.startswith("data:image/") or len(data_url) > 15_000_000:\n        raise ValueError("請選擇 10MB 以內的 PNG、JPEG 或 WebP 圖片。")\n    raw = base64.b64decode(data_url.split(",", 1)[1], validate=True)\n    if len(raw) > 10 * 1024**2:\n        raise ValueError("圖片請限制在 10MB 以內。")\n    with Image.open(io.BytesIO(raw)) as source:\n        if source.width * source.height > 30_000_000:\n            raise ValueError("圖片請限制在 3,000 萬像素以內。")\n        picture = ImageOps.exif_transpose(source).convert("RGB")\n        picture.thumbnail((2048, 2048))\n        name = "h3ref_" + uuid.uuid4().hex + ".png"\n        target = COMFY / "input" / name\n        target.parent.mkdir(parents=True, exist_ok=True)\n        picture.save(target)\n    return {"filename": name, "width": picture.width, "height": picture.height}\n\n\ndef _add_refine_pass(graph, config, sampler_name, prefix):\n    """Second sampling at 720p: decode draft, resize, re-encode, then run only the tail of the same schedule (8 steps for omni_hd, `refine_steps` of `steps` for ref2va_hd).\n    The distilled model was trained on those exact timesteps, so a tail split beats a partial-denoise schedule."""\n    rw, rh, cw, ch, cx, cy = HD_TARGETS[config["size"]]\n    graph["40"] = {"class_type": "LTXVSeparateAVLatent", "inputs": {"av_latent": ["12", 0]}}\n    graph["41"] = {"class_type": "VAEDecode", "inputs": {"samples": ["40", 0], "vae": ["4", 0]}}\n    graph["42"] = {"class_type": "ImageScale", "inputs": {"image": ["41", 0], "upscale_method": "lanczos", "width": rw, "height": rh, "crop": "disabled"}}\n    graph["43"] = {"class_type": "VAEEncode", "inputs": {"pixels": ["42", 0], "vae": ["4", 0]}}\n    graph["44"] = {"class_type": "LTXVConcatAVLatent", "inputs": {"video_latent": ["43", 0], "audio_latent": ["40", 1]}}\n    refs = {"clip": ["3", 0], "vae": ["4", 0], "audio_vae": ["5", 0], "prompt": config["prompt"], "width": rw, "height": rh, "length": config["frames"], "ref_image_size": "match"}\n    for index in range(len(config["refs"])):\n        refs[f"ref_images.ref_image_{index}"] = [str(30 + index), 0]\n    graph["45"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": refs}\n    graph["46"] = {"class_type": "BasicGuider", "inputs": {"model": ["16", 0] if "16" in graph else ["1", 0], "conditioning": ["45", 0]}}\n    graph["48"] = {"class_type": "SplitSigmas", "inputs": {"sigmas": ["11", 0], "step": config["steps"] - config["refine_steps"]}}\n    graph["49"] = {"class_type": "RandomNoise", "inputs": {"noise_seed": (config["seed"] + 1) % 2**32}}\n    graph["50"] = {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["49", 0], "guider": ["46", 0], "sampler": ["10", 0], "sigmas": ["48", 1], "latent_image": ["44", 0]}}\n    graph["13"]["inputs"]["samples"] = ["50", 0]\n    graph["14"]["inputs"]["samples"] = ["50", 0]\n    graph["51"] = {"class_type": "ImageCrop", "inputs": {"image": ["13", 0], "width": cw, "height": ch, "x": cx, "y": cy}}\n    graph["15"]["inputs"]["images"] = ["51", 0]\n    # silent draft preview, to judge draft vs final and to confirm a seed reproduces the draft you picked\n    graph["60"] = {"class_type": "VHS_VideoCombine", "inputs": {"images": ["41", 0], "frame_rate": 24, "loop_count": 0, "filename_prefix": prefix + "_draft", "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": 18, "save_metadata": False, "trim_to_audio": False, "pingpong": False, "save_output": True}}\n\n\ndef _add_ayase_pass(graph, config, prefix):\n    """Ayase Studio second sampling: split the AV latent, neural-upscale the video half, re-noise with a new seed and run a\n    partial-denoise pass at the target size (BasicScheduler denoise = the studio\'s sigma tail), then put the first-pass audio back."""\n    w2, h2 = config["target_width"], config["target_height"]\n    graph["40"] = {"class_type": "LTXVSeparateAVLatent", "inputs": {"av_latent": ["12", 0]}}\n    graph["41"] = {"class_type": "MinimaxH3LatentUpscaler3D", "inputs": {"latent": ["40", 0], "model_name": AYASE_UPSCALER_NAME, "mode": "target dimensions", "mode.width": w2, "mode.height": h2,\n                   "align": 32, "enable_temporal_chunking": True, "force_unload": True, "device": "cuda", "precision": "fp16"}}\n    graph["42"] = {"class_type": "LTXVConcatAVLatent", "inputs": {"video_latent": ["41", 0], "audio_latent": ["40", 1]}}\n    refs = {"clip": ["3", 0], "vae": ["4", 0], "audio_vae": ["5", 0], "prompt": config["prompt"] + "\\n\\nrepair_instruction:\\n" + AYASE_REPAIR, "width": w2, "height": h2, "length": config["frames"], "ref_image_size": "match"}\n    for index in range(len(config["refs"])):\n        refs[f"ref_images.ref_image_{index}"] = [str(30 + index), 0]\n    graph["45"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": refs}\n    graph["46"] = {"class_type": "BasicGuider", "inputs": {"model": ["16", 0], "conditioning": ["45", 0]}}\n    graph["47"] = {"class_type": "BasicScheduler", "inputs": {"model": ["16", 0], "scheduler": "simple", "steps": config["second_steps"], "denoise": config["second_denoise"]}}\n    graph["49"] = {"class_type": "RandomNoise", "inputs": {"noise_seed": (config["seed"] + 1) % 2**32}}\n    if config.get("tiled"):\n        # Same pack as the upscaler: spatial tiles (the first-pass size) and 243-frame temporal chunks with overlap, as in the user\'s local single-segment workflow.\n        graph["55"] = {"class_type": "MMH3TemporalSplitParamsV10", "inputs": {"chunk_frames": 243, "temporal_overlap_frames": 22, "anchor_strength": 0.999, "motion_anchor_frames": "22", "identity_anchor_frames": 24}}\n        graph["56"] = {"class_type": "MMH3SpatialSplitParamsV10", "inputs": {"tile_width": config["width"], "tile_height": config["height"], "overlap_ratio": 0.25, "fade_ratio": 0.5, "min_tile_size": 256, "seam_denoise": 0.7}}\n        graph["50"] = {"class_type": "MMH3SplitUpscale", "inputs": {"model": ["16", 0], "conditioning": ["45", 0], "latent": ["42", 0], "noise": ["49", 0], "sampler": ["10", 0], "sigmas": ["47", 0], "cfg": 1.0,\n                       "temporal_split_param": ["55", 0], "spatial_split_param": ["56", 0], "seam_polish": "auto", "color_match": True}}\n    else:\n        graph["50"] = {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["49", 0], "guider": ["46", 0], "sampler": ["10", 0], "sigmas": ["47", 0], "latent_image": ["42", 0]}}\n    graph["52"] = {"class_type": "LTXVSeparateAVLatent", "inputs": {"av_latent": ["50", 0]}}\n    graph["53"] = {"class_type": "LTXVConcatAVLatent", "inputs": {"video_latent": ["52", 0], "audio_latent": ["40", 1]}}\n    graph["13"]["inputs"]["samples"] = ["53", 0]\n    graph["14"]["inputs"]["samples"] = ["53", 0]\n    graph["54"] = {"class_type": "VAEDecode", "inputs": {"samples": ["40", 0], "vae": ["4", 0]}}\n    graph["60"] = {"class_type": "VHS_VideoCombine", "inputs": {"images": ["54", 0], "frame_rate": 24, "loop_count": 0, "filename_prefix": prefix + "_draft", "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": 18, "save_metadata": False, "trim_to_audio": False, "pingpong": False, "save_output": True}}\n\n\ndef graph_for(config, profile="int8", prefix="h3"):\n    files = {k: Path(v).name for k, v in PROFILES[profile]["files"].items()}\n    if config["mode"] in ("omni", "omni_hd"):\n        files["model"] = Path(OMNI_FILES[profile]).name\n    if config["mode"] in ("ref4", "ref2va", "ref2va_hd", "ayase"):\n        files["model"], files["lora"] = (Path(v).name for v in REF4_FILES.values())\n    if config["mode"] == "ayase":\n        files["lora"] = AYASE_LORA[2]\n    sampler_name, scheduler = SAMPLERS[config.get("sampler", "res_multistep+simple")]\n    graph = {\n        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": files["model"], "weight_dtype": "default"}},\n        "3": {"class_type": "CLIPLoader", "inputs": {"clip_name": files["text"], "type": "minimax", "device": "default"}},\n        "4": {"class_type": "VAELoader", "inputs": {"vae_name": files["video"]}},\n        "5": {"class_type": "VAELoader", "inputs": {"vae_name": files["audio"]}},\n        "7": {"class_type": "MiniMaxH3ImageToVideo", "inputs": {"clip": ["3", 0], "vae": ["4", 0], "prompt": config["prompt"], "width": config["width"], "height": config["height"], "length": config["frames"]}},\n        "8": {"class_type": "BasicGuider", "inputs": {"model": ["1", 0], "conditioning": ["7", 0]}},\n        "9": {"class_type": "RandomNoise", "inputs": {"noise_seed": config["seed"]}},\n        "10": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": sampler_name}},\n        "11": {"class_type": "BasicScheduler", "inputs": {"model": ["1", 0], "scheduler": scheduler, "steps": config["steps"], "denoise": 1.0}},\n        "12": {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["9", 0], "guider": ["8", 0], "sampler": ["10", 0], "sigmas": ["11", 0], "latent_image": ["7", 1]}},\n        "13": {"class_type": "VAEDecode", "inputs": {"samples": ["12", 0], "vae": ["4", 0]}},\n        "14": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["12", 0], "vae": ["5", 0]}},\n        "15": {"class_type": "VHS_VideoCombine", "inputs": {"images": ["13", 0], "audio": ["14", 0], "frame_rate": 24, "loop_count": 0, "filename_prefix": prefix, "format": "video/h264-mp4", "pix_fmt": "yuv420p", "crf": 18, "save_metadata": True, "trim_to_audio": False, "pingpong": False, "save_output": True}},\n    }\n    if config["mode"] in ("turbo", "turbo4"):  # exactly one turbo LoRA; never stack two\n        graph["2"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": files["lora" if config["mode"] == "turbo" else "lora4"], "strength_model": 1.0}}\n        graph["8"]["inputs"]["model"] = ["2", 0]\n        graph["11"]["inputs"]["model"] = ["2", 0]\n    if config["mode"] == "ref4":  # official ref2v template path: the 4-step turbo LoRA only, no extra shift node\n        graph["2"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": files["lora"], "strength_model": 1.0}}\n        graph["8"]["inputs"]["model"] = ["2", 0]\n        graph["11"]["inputs"]["model"] = ["2", 0]\n    if config["mode"] == "ayase":  # HyperFlow LoRA, then video/audio shift 12/3 (the sigma grid HyperFlow was trained on)\n        graph["2"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["1", 0], "lora_name": files["lora"], "strength_model": 1.0}}\n        graph["16"] = {"class_type": "MiniMaxH3SigmaShift", "inputs": {"model": ["2", 0], "shift_video": 12.0, "shift_audio": 3.0}}\n        graph["8"]["inputs"]["model"] = ["16", 0]\n        graph["11"]["inputs"]["model"] = ["16", 0]\n    if config["mode"] in ("omni", "omni_hd"):\n        # FastH3 Omni is already 8-step distilled: no turbo LoRA, video/audio shift 12/3 as its README says.\n        graph["16"] = {"class_type": "MiniMaxH3SigmaShift", "inputs": {"model": ["1", 0], "shift_video": 12.0, "shift_audio": 3.0}}\n        graph["8"]["inputs"]["model"] = ["16", 0]\n        graph["11"]["inputs"]["model"] = ["16", 0]\n    if config.get("shift"):  # opt-in video/audio shift 12/3: community Ref2VA workflows and Ayase Studio use it, the official r2v template does not\n        graph["16"] = {"class_type": "MiniMaxH3SigmaShift", "inputs": {"model": graph["8"]["inputs"]["model"], "shift_video": 12.0, "shift_audio": 3.0}}\n        graph["8"]["inputs"]["model"] = ["16", 0]\n        graph["11"]["inputs"]["model"] = ["16", 0]\n    # ref2va = official Comfy-Org r2v template: non-distilled Ref2VA, no LoRA, no shift node (unless shift=true), BasicGuider, res_multistep + simple\n    if config["mode"] in REF_MODES:\n        graph["7"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": {"clip": ["3", 0], "vae": ["4", 0], "audio_vae": ["5", 0], "prompt": config["prompt"], "width": config["width"], "height": config["height"], "length": config["frames"], "ref_image_size": "match"}}\n        for index, name in enumerate(config["refs"]):\n            graph[str(30 + index)] = {"class_type": "LoadImage", "inputs": {"image": name}}\n            graph["7"]["inputs"][f"ref_images.ref_image_{index}"] = [str(30 + index), 0]\n    if config["mode"] in HD_MODES:\n        _add_refine_pass(graph, config, sampler_name, prefix)\n    if config["mode"] == "ayase":\n        _add_ayase_pass(graph, config, prefix)\n    for number, key in (("20", "first_frame"), ("21", "last_frame")):\n        if config.get(key):\n            graph[number] = {"class_type": "LoadImage", "inputs": {"image": config[key]}}\n            graph["7"]["inputs"][key] = [number, 0]\n    return graph\n\n\ndef _update(job_id, **values):\n    with _lock:\n        _jobs[job_id].update(values)\n\n\ndef _progress_event(job_id, prompt_id, event):\n    data = event.get("data", {})\n    if data.get("prompt_id") != prompt_id:\n        return\n    if event.get("type") == "executing":\n        node = str(data.get("node"))\n        if node != "None":\n            with _lock:  # per-node start times, so draft / refine / decode can be timed separately\n                _jobs[job_id].setdefault("timeline", []).append([node, round(time.time() - _jobs[job_id]["started"], 1)])\n        stage, message = {\n            "1": ("loading", "載入模型中"), "3": ("loading", "載入模型中"),\n            "7": ("loading", "載入提示詞中"), "12": ("sampling", "生成影片與音訊中"),\n            "40": ("sampling", "放大草稿並準備精修中"), "45": ("sampling", "放大草稿並準備精修中"), "50": ("sampling", "二次取樣精修中"),\n            "13": ("saving", "輸出影片中"), "14": ("saving", "輸出音訊中"),\n            "15": ("saving", "儲存為 MP4 中"),\n        }.get(node, (None, None))\n        if stage:\n            _update(job_id, stage=stage, message=message, step=None, total_steps=None)\n    elif event.get("type") == "progress" and str(data.get("node")) in ("12", "50"):\n        value, total = data.get("value"), data.get("max")\n        if isinstance(value, (int, float)) and isinstance(total, (int, float)) and total > 0:\n            _update(job_id, stage="sampling", message="生成影片與音訊中", step=min(value, total), total_steps=total)\n\n\ndef _listen_progress(socket, stopped, job_id, prompt_id):\n    import websocket\n    while not stopped.is_set():\n        try:\n            message = socket.recv()\n            if isinstance(message, str) and message:\n                _progress_event(job_id, prompt_id, json.loads(message))\n        except websocket.WebSocketTimeoutException:\n            continue\n        except Exception:\n            return  # History polling still completes the job if the socket closes.\n\n\ndef submit(config):\n    global _active\n    config = clean_config(config)\n    if not (BASE / "setup.json").exists():\n        raise RuntimeError("請先執行「準備環境」儲存格。")\n    if config["mode"] in ("omni", "omni_hd") and not json.loads((BASE / "setup.json").read_text()).get("omni"):\n        raise RuntimeError("尚未下載 FastH3 Omni 模型。請在 ① 勾選「下載 FastH3 Omni」後重新執行。")\n    record = json.loads((BASE / "setup.json").read_text())\n    if config["mode"] in ("ref4", "ref2va", "ref2va_hd", "ayase") and not record.get("ref4"):\n        raise RuntimeError("尚未下載 Ref2VA 模型（ref4／ref2va 共用）。請在 ① 勾選「download_ref4」後重新執行。")\n    with _lock:\n        if _active:\n            raise RuntimeError("目前正在生成。請等待完成或中止後再執行。")\n        job_id = uuid.uuid4().hex[:12]\n        _active = job_id\n        _jobs[job_id] = {"id": job_id, "state": "running", "stage": "loading", "message": "連線到生成引擎中", "started": time.time(), "config": config, "cancel": False}\n    threading.Thread(target=_worker, args=(job_id, config), daemon=True).start()\n    return {"id": job_id}\n\n\ndef _comfy_error(messages):\n    """Readable ComfyUI failure: node, exception type/message and traceback tail. The raw dump holds huge tensor prints."""\n    for kind, info in messages:\n        if kind == "execution_error" and isinstance(info, dict):\n            trace = [str(line).strip()[:300] for line in info.get("traceback", [])][-6:]\n            head = f"節點 {info.get(\'node_id\')} ({info.get(\'node_type\')}) 失敗：{info.get(\'exception_type\')}: {str(info.get(\'exception_message\'))[:800]}"\n            return head + chr(10) + chr(10).join(trace)\n    return json.dumps(messages, ensure_ascii=False)[:1500]\n\n\ndef _worker(job_id, config):\n    global _active\n    started = time.monotonic()\n    folder = OUTPUT / job_id\n    folder.mkdir(parents=True, exist_ok=True)\n    socket = None\n    listener = None\n    stopped = threading.Event()\n    try:\n        record = json.loads((BASE / "setup.json").read_text())\n        start_server(record.get("omni"))\n        graph = graph_for(config, record["profile"], f"h3_lab/{job_id}")\n        client_id = "h3-lab-" + job_id\n        try:\n            import websocket\n            socket = websocket.create_connection(f"ws://127.0.0.1:{PORT}/ws?clientId={client_id}", timeout=1)\n        except Exception:\n            socket = None\n        response = _request("POST", "/prompt", json={"prompt": graph, "client_id": client_id})\n        if response.get("node_errors") or "prompt_id" not in response:\n            raise RuntimeError(json.dumps(response, ensure_ascii=False))\n        prompt_id = response["prompt_id"]\n        _update(job_id, message="載入並生成模型中（首次較久）", prompt_id=prompt_id)\n        if socket:\n            listener = threading.Thread(target=_listen_progress, args=(socket, stopped, job_id, prompt_id), daemon=True)\n            listener.start()\n        min_free = None\n        busy_since = None\n        while time.monotonic() - started < 5400:\n            try:  # lowest free VRAM seen during the job shows directly whether a long/HD clip fits this GPU\n                device = _request("GET", "/system_stats", timeout=5)["devices"][0]\n                free = device["torch_vram_free"] if "torch_vram_free" in device else device["vram_free"]\n                min_free = free if min_free is None else min(min_free, free)\n                vram_total = device["vram_total"]\n                _update(job_id, min_vram_free_gib=round(min_free / 2**30, 1), vram_total_gib=round(vram_total / 2**30, 1))\n            except Exception:\n                pass\n            if _jobs[job_id]["cancel"]:\n                _request("POST", "/interrupt")\n                raise InterruptedError("已中止生成。")\n            try:  # a long decode/upscale can keep the server from answering for a while; that is not a failure unless it exits\n                history = _request("GET", "/history/" + prompt_id, timeout=90)\n                busy_since = None\n            except Exception as exc:\n                import requests\n                if not isinstance(exc, requests.exceptions.RequestException):\n                    raise\n                if _process is not None and _process.poll() is not None:\n                    raise RuntimeError("生成引擎已結束（可能是記憶體不足）。" + (BASE / "comfy.log").read_text(errors="replace")[-1500:])\n                busy_since = busy_since or time.monotonic()\n                if time.monotonic() - busy_since > 900:\n                    raise RuntimeError("生成引擎超過 15 分鐘沒有回應：" + str(exc))\n                _update(job_id, message="生成引擎忙碌中，等待回應")\n                time.sleep(5)\n                continue\n            result = history.get(prompt_id)\n            if result:\n                if result.get("status", {}).get("status_str") == "error":\n                    raise RuntimeError(_comfy_error(result.get("status", {}).get("messages", [])))\n                outputs = result.get("outputs", {}).get("15", {}).get("gifs", [])\n                video = next((x for x in outputs if x.get("filename", "").endswith(".mp4")), None)\n                if video:\n                    break\n                if result.get("status", {}).get("completed"):\n                    raise RuntimeError("處理已結束，但未回傳影片檔。")\n            time.sleep(2)\n        else:\n            _request("POST", "/interrupt")\n            raise TimeoutError("已達 90 分上限，因此中止此次生成。")\n        stopped.set()\n        if listener:\n            listener.join(timeout=2)\n        _update(job_id, stage="saving", message="確認並儲存影片中", step=None, total_steps=None)\n        source = (COMFY / "output" / video.get("subfolder", "") / video["filename"]).resolve()\n        if not source.is_relative_to((COMFY / "output").resolve()):\n            raise RuntimeError("輸出路徑不正確。")\n        shutil.copy2(source, folder / "video.mp4")\n        draft = next((x for x in result.get("outputs", {}).get("60", {}).get("gifs", []) if x.get("filename", "").endswith(".mp4")), None)\n        if draft:\n            shutil.copy2(COMFY / "output" / draft.get("subfolder", "") / draft["filename"], folder / "draft.mp4")\n        import imageio_ffmpeg\n        ffmpeg = shutil.which("ffmpeg") or imageio_ffmpeg.get_ffmpeg_exe()\n        # Decode one frame of each stream so an empty/broken result is not reported as success.\n        for stream in ("v", "a"):\n            check = subprocess.run([ffmpeg, "-v", "error", "-i", str(folder / "video.mp4"), "-map", f"0:{stream}:0", f"-frames:{stream}", "1", "-f", "null", "-"], capture_output=True, text=True, timeout=90)\n            if check.returncode:\n                raise RuntimeError("無法確認生成檔案的影片或音訊。" + check.stderr[-1200:])\n        (folder / "prompt.txt").write_text(config["prompt"], encoding="utf-8")\n        (folder / "workflow-api.json").write_text(json.dumps(graph, ensure_ascii=False, indent=2))\n        report = {"id": job_id, "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "generation_seconds": round(time.monotonic() - started, 2), "config": config, "environment": record, "video_bytes": (folder / "video.mp4").stat().st_size, "video_audio_decode_check": True,\n                  "timeline": _jobs[job_id].get("timeline", []), "min_vram_free_gib": round(min_free / 2**30, 1) if min_free is not None else None, "vram_total_gib": round(vram_total / 2**30, 1) if min_free is not None else None}\n        (folder / "settings.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))\n        copies = {k: config[k] for k in ("first_frame", "last_frame") if config.get(k)}\n        copies.update({f"ref_{i + 1}": name for i, name in enumerate(config.get("refs", []))})\n        for key, name in copies.items():\n            shutil.copy2(COMFY / "input" / name, folder / (key + ".png"))\n        with zipfile.ZipFile(folder / "result.zip", "w", zipfile.ZIP_DEFLATED) as bundle:\n            for item in folder.iterdir():\n                if item.name != "result.zip":\n                    bundle.write(item, item.name)\n        _update(job_id, state="done", stage="done", message="已完成", report=report)\n    except InterruptedError as exc:\n        _update(job_id, state="cancelled", message=str(exc))\n    except Exception as exc:\n        hint = " 若記憶體不足，請以 5 秒・橫式・Turbo 重試。" if "memory" in str(exc).lower() else ""\n        seen = _jobs[job_id]\n        vram = f" [VRAM 最低剩餘 {seen[\'min_vram_free_gib\']} / {seen[\'vram_total_gib\']} GiB]" if seen.get("min_vram_free_gib") is not None else ""\n        _update(job_id, state="error", message=str(exc) + hint + vram)\n    finally:\n        stopped.set()\n        if socket:\n            socket.close()\n        if listener:\n            listener.join(timeout=2)\n        with _lock:\n            _jobs[job_id]["ended"] = time.time()\n            _active = None\n\n\ndef status(job_id):\n    with _lock:\n        if job_id not in _jobs:\n            raise ValueError("找不到此生成。請從歷史紀錄確認結果。")\n        state = dict(_jobs[job_id])\n    state.pop("cancel", None)\n    state["elapsed"] = round(state.get("ended", time.time()) - state["started"])\n    return state\n\n\ndef cancel(job_id):\n    with _lock:\n        if job_id != _active:\n            return {"message": "目前沒有執行中的生成。"}\n        _jobs[job_id]["cancel"] = True\n    return {"message": "已要求中止，請稍候至處理結束。"}\n\n\ndef download(job_id, kind="video"):\n    if not job_id.isalnum() or len(job_id) != 12:\n        raise ValueError("生成 ID 不正確。")\n    names = {"video": ("video.mp4", "video/mp4"), "zip": ("result.zip", "application/zip"), "settings": ("settings.json", "application/json")}\n    if kind not in names:\n        raise ValueError("儲存格式不正確。")\n    name, mime = names[kind]\n    path = OUTPUT / job_id / name\n    if not path.is_file():\n        raise ValueError("找不到結果。刪除執行階段後結果也會一併消失。")\n    if path.stat().st_size > 60 * 1024**2:\n        raise ValueError("檔案超過 60MB，請從 Colab 左側檔案清單開啟 /content/h3-lab/results 後再儲存。")\n    return {"filename": f"minimax_h3_{job_id}.{name.split(\'.\')[-1]}", "mime": mime, "base64": base64.b64encode(path.read_bytes()).decode()}\n\n\ndef about():\n    record = json.loads((BASE / "setup.json").read_text()) if (BASE / "setup.json").exists() else {}\n    results = []\n    if OUTPUT.exists():\n        for file in sorted(OUTPUT.glob("*/settings.json"), key=lambda p: p.stat().st_mtime, reverse=True)[:20]:\n            try:\n                results.append(json.loads(file.read_text()))\n            except (ValueError, OSError):\n                pass\n    return {"ready": bool(record), "environment": record, "results": results, "active": _active}\n\n\ndef install_callbacks():\n    from google.colab import output\n    from IPython.display import JSON\n    def safe(function):\n        def call(*args):\n            try:\n                result = {"ok": True, "value": function(*args)}\n            except Exception as exc:\n                result = {"ok": False, "error": str(exc)}\n            # invokeFunction returns an IPython MIME bundle, not the raw dict.\n            return JSON(result)\n        return call\n    for name, fn in {"about": about, "upload": upload_image, "run": submit, "status": status, "cancel": cancel, "download": download}.items():\n        output.register_callback("h3lab." + name, safe(fn))\n\n\n# ---- Agent mode: Google Drive folder queue (no tunnel, no public port) ----\nimport re as _re\n\nQUEUE_DIRS = ("inbox", "running", "done", "failed")\n_ID = _re.compile(r"^[A-Za-z0-9_-]{1,64}$")\nIMAGE_KEYS = {key: f".{key}.png" for key in IMAGE_NAMES}\nIMAGE_WAIT_SECONDS = 600  # Drive uploads are async: wait this long for declared images, then fail\nHEARTBEAT_SECONDS = 300\n\n\ndef queue_init(root):\n    root = Path(root)\n    for name in QUEUE_DIRS:\n        (root / name).mkdir(parents=True, exist_ok=True)\n    return root\n\n\ndef _write_json(path, value):\n    # Write to a temp name then rename, so an agent never reads a half-written file.\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef queue_recover(root):\n    """Sessions die at 12/24h: put jobs left in running/ back, unless already finished."""\n    root = Path(root)\n    for file in (root / "running").glob("*.json"):\n        if (root / "done" / file.name).exists() or (root / "failed" / file.name).exists():\n            file.unlink()\n        else:\n            os.replace(file, root / "inbox" / file.name)\n\n\ndef _fail(root, job_id, message):\n    _write_json(Path(root) / "failed" / f"{job_id}.json", {"id": job_id, "state": "failed", "error": message})\n\n\ndef _declared_images(config):\n    declared = config.get("images", []) if isinstance(config, dict) else []\n    return [k for k in declared if k in IMAGE_KEYS] if isinstance(declared, list) else []\n\n\ndef process_one(root, runner, validate):\n    """Claim and run one inbox job. Returns the id handled, or None when nothing is ready."""\n    root = Path(root)\n    for file in sorted((root / "inbox").glob("*.json"), key=lambda p: p.stat().st_mtime):\n        job_id = file.stem\n        if not _ID.match(job_id):\n            file.unlink()\n            continue\n        if (root / "done" / file.name).exists():\n            file.unlink()  # duplicate submission: result already exists\n            continue\n        try:\n            peek = json.loads(file.read_text(encoding="utf-8"))\n        except (ValueError, OSError):\n            peek = None  # malformed: claim it below so it lands in failed/\n        missing = [k for k in _declared_images(peek) if not (root / "inbox" / (job_id + IMAGE_KEYS[k])).is_file()]\n        if missing and time.time() - file.stat().st_mtime < IMAGE_WAIT_SECONDS:\n            continue  # reference image still syncing from Drive\n        claimed = root / "running" / file.name\n        try:\n            os.replace(file, claimed)\n        except OSError:\n            continue\n        requeue = False\n        try:\n            if missing:\n                raise ValueError("參考圖片沒有送達：" + ", ".join(missing))\n            config = json.loads(claimed.read_text(encoding="utf-8"))\n            if not isinstance(config, dict):\n                raise ValueError("工作單必須是 JSON 物件。")\n            for key in (*IMAGE_KEYS, "images"):\n                config.pop(key, None)  # images come only from sibling files, never from raw paths in the JSON\n            config = validate(config)\n            images = {key: root / "inbox" / (job_id + suffix) for key, suffix in IMAGE_KEYS.items() if (root / "inbox" / (job_id + suffix)).is_file()}\n            video, settings = runner(config, images)\n            shutil.copy2(video, root / "done" / f"{job_id}.mp4")\n            shutil.copy2(settings, root / "done" / f"{job_id}.settings.json")\n            draft = Path(video).with_name("draft.mp4")  # omni_hd keeps its silent draft so draft vs final can be compared\n            if draft.is_file():\n                shutil.copy2(draft, root / "done" / f"{job_id}.draft.mp4")\n            # Marker goes last and carries the mp4 size: Drive may sync the marker before the video.\n            _write_json(root / "done" / f"{job_id}.json", {"id": job_id, "state": "done", "video": f"{job_id}.mp4", "video_bytes": (root / "done" / f"{job_id}.mp4").stat().st_size, "settings": f"{job_id}.settings.json", "finished_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})\n        except Exception as exc:\n            _fail(root, job_id, str(exc)[-3000:])\n        except BaseException:\n            requeue = True  # Ctrl-C / runtime kill: keep the job\n            raise\n        finally:\n            if requeue:\n                os.replace(claimed, root / "inbox" / file.name)\n            else:\n                claimed.unlink(missing_ok=True)\n                for suffix in IMAGE_KEYS.values():\n                    (root / "inbox" / (job_id + suffix)).unlink(missing_ok=True)\n        return job_id\n    return None\n\n\ndef comfy_runner(config, images):\n    """Real runner: upload reference images (PIL-validated), submit, block until the job ends."""\n    for key, path in images.items():\n        data = base64.b64encode(path.read_bytes()).decode()\n        config[key] = upload_image("data:image/png;base64," + data)["filename"]\n    job_id = submit(config)["id"]\n    try:\n        while True:\n            state = status(job_id)\n            if state["state"] != "running":\n                break\n            time.sleep(3)\n    except BaseException:\n        cancel(job_id)  # stop ComfyUI so the next run is not blocked by a ghost job\n        raise\n    if state["state"] != "done":\n        raise RuntimeError(state.get("message", "生成失敗。"))\n    folder = OUTPUT / job_id\n    return folder / "video.mp4", folder / "settings.json"\n\n\ndef agent_loop(root, poll=20, runner=None, validate=None, max_loops=None):\n    """Blocking queue worker. Keep this cell running; create <root>/STOP to end it."""\n    root = queue_init(root)\n    runner = runner or comfy_runner\n    validate = validate or (lambda c: clean_config(c, need_refs=False))  # images are attached after validation\n    (root / "STOP").unlink(missing_ok=True)  # a leftover STOP from the last run must not end this one\n    while _active:  # a cancelled job may still be winding down\n        time.sleep(2)\n    queue_recover(root)\n    loops, beat = 0, 0.0\n    while not (root / "STOP").exists() and (max_loops is None or loops < max_loops):\n        loops += 1\n        handled = process_one(root, runner, validate)\n        if handled or time.time() - beat >= HEARTBEAT_SECONDS:\n            _write_json(root / "heartbeat.json", {"alive_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "last_job": handled})\n            beat = time.time()\n        if handled:\n            print(time.strftime("%H:%M:%S"), "完成處理工作", handled)\n        else:\n            time.sleep(poll)\n'
_module_name = 'minimax_h3_colab_backend_v3'
_source_hash = hashlib.sha256(BACKEND_SOURCE.encode()).hexdigest()
h3 = sys.modules.get(_module_name)
if h3 is None or getattr(h3, '_source_hash', None) != _source_hash:
    if h3 is not None and getattr(h3, '_active', None):
        raise RuntimeError('請等目前的生成結束後，再重新執行這個 cell。')
    h3 = types.ModuleType(_module_name)
    exec(compile(BACKEND_SOURCE, 'h3_backend.py', 'exec'), h3.__dict__)
    h3._source_hash = _source_hash
    sys.modules[_module_name] = h3
UI_SOURCE = '<!doctype html>\n<html lang="zh-Hant-TW"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><link rel="icon" href="data:,">\n<style>\n:root{color-scheme:light;--ink:#18233b;--muted:#68758d;--line:#dce3ef;--accent:#4c48d9;--soft:#f4f5fd;--ok:#14765e}*{box-sizing:border-box}body{margin:0;background:#f6f8fc;color:var(--ink);font:14px/1.65 -apple-system,BlinkMacSystemFont,"Noto Sans JP",sans-serif}button,input,textarea,select{font:inherit}button{cursor:pointer;transition:.15s;background:white;border:1px solid var(--line);border-radius:9px;padding:9px 13px;color:var(--ink)}button:hover{border-color:#8d8adf;background:#f5f4ff}button:disabled{opacity:.5;cursor:wait}button.primary{background:var(--accent);color:white;border-color:var(--accent);font-weight:700}button.primary:hover{background:#3e3bc1}button.ghost{background:transparent}button.small{font-size:12px;padding:6px 10px}input,textarea,select{width:100%;border:1px solid #cbd5e4;border-radius:8px;background:white;color:var(--ink);padding:10px}textarea{resize:vertical;line-height:1.55}textarea.prompt{font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-size:13px;min-height:150px}label{display:block;font-size:12px;font-weight:650;margin-bottom:6px}input:focus,textarea:focus,select:focus,button:focus-visible{outline:3px solid #a8a5ff;outline-offset:2px}.wrap{max-width:1160px;margin:auto;padding:28px}.eyebrow{font-size:11px;color:var(--accent);font-weight:750;letter-spacing:2px}h1{font-size:27px;line-height:1.3;margin:5px 0 7px;letter-spacing:-.6px}h2{font-size:17px;margin:0}p{margin:5px 0 12px}.sub,.hint{color:var(--muted);font-size:12px}.top{display:flex;justify-content:space-between;align-items:center;gap:15px;margin-bottom:23px}.pill{display:inline-block;background:#e9f5ef;color:var(--ok);padding:6px 10px;border-radius:20px;font-size:11px}.pill.pending{background:#fff2d6;color:#80600a}.grid{display:grid;grid-template-columns:minmax(0,1.15fr) minmax(300px,.85fr);gap:20px}.card{background:white;border:1px solid var(--line);border-radius:14px;padding:21px;margin-bottom:18px;box-shadow:0 3px 12px #14234b03}.head{display:flex;align-items:center;justify-content:space-between;gap:10px;margin-bottom:16px}.step{color:var(--accent);margin-right:8px}.row{display:flex;gap:9px;align-items:center;flex-wrap:wrap}.row>*{flex:1;min-width:0}.row button{flex:0 0 auto}.cols{display:grid;grid-template-columns:1fr 1fr;gap:13px}.field{margin-bottom:14px}.field:last-child{margin-bottom:0}.space{margin-top:13px}.divider{border-top:1px solid var(--line);margin:19px 0}details{border:1px solid var(--line);border-radius:9px;padding:11px 13px;margin-top:14px}summary{font-size:13px;cursor:pointer;font-weight:650}details .inside{padding-top:15px}.note{border-left:3px solid #bbb8f5;padding:8px 11px;background:var(--soft);font-size:12px;color:#505975;border-radius:0 7px 7px 0}.hidden{display:none!important}.actions{display:flex;gap:8px;flex-wrap:wrap;margin-top:10px}.actions button{font-size:12px}.reference{background:#fafbfe;border:1px dashed #c8d1e2;border-radius:9px;padding:11px}.reference img{max-width:100%;height:90px;object-fit:contain;display:block;margin:8px auto}.reference input{font-size:11px;padding:7px}.reference .hint{font-size:11px}.duration{color:var(--muted);font-size:11px;margin-top:6px}.run{width:100%;font-size:15px;padding:13px;margin-top:16px}.status{padding:14px;border-radius:9px;background:#f4f6fc;margin-top:12px;font-size:12px;white-space:pre-wrap;overflow-wrap:anywhere}.status.error{background:#fff0f0;color:#922c36}.status.success{background:#eaf7f0;color:#155d48}.status.running{border-left:3px solid var(--accent)}.preview{min-height:185px;display:flex;flex-direction:column;justify-content:center;text-align:center;background:#f4f6fa;border:1px dashed #cbd5e4;border-radius:10px;color:var(--muted);padding:25px}.preview .icon{font-size:30px;color:#b4bdd2}video{display:block;width:100%;border-radius:10px;background:#111828;max-height:410px}.resultmeta{font-size:11px;color:var(--muted);margin-top:8px}.historyitem{border-top:1px solid var(--line);padding-top:12px;margin-top:12px}.historyitem p{font-size:12px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;margin-bottom:6px}.historyitem .actions{margin-top:4px}.foot{color:var(--muted);font-size:11px;margin:4px 0 20px}.toast{position:fixed;bottom:18px;left:50%;transform:translateX(-50%);background:#202942;color:white;padding:10px 20px;border-radius:9px;max-width:90%;box-shadow:0 5px 20px #0002;z-index:10}.count{text-align:right;font-size:11px;color:var(--muted)}.compact{font-size:11px}.full{grid-column:1/-1}.inline-note{font-size:11px;color:var(--muted);margin-top:8px}.recording .foot{display:none}@media(max-width:760px){.wrap{padding:16px}.grid{grid-template-columns:1fr}.top{align-items:flex-start;flex-direction:column}.card{padding:16px}.cols{gap:10px}h1{font-size:23px}}@media(max-width:390px){.cols{grid-template-columns:1fr}.actions button{flex:1 0 45%;white-space:nowrap}.wrap{padding:10px}}\n.estimate{text-align:center;color:var(--muted);font-size:12px;margin:7px 0 0}.stages{display:flex;gap:6px;margin:12px 0 8px}.stages span{flex:1;text-align:center;background:#f1f3f8;border-radius:5px;color:var(--muted);font-size:11px;padding:4px}.stages .active{background:#eceafd;color:var(--accent);font-weight:700}.stages .done{background:#e9f5ef;color:var(--ok)}progress{display:block;width:100%;height:8px;accent-color:var(--accent)}.progressMeta{display:flex;justify-content:space-between;color:var(--muted);font-size:11px;margin-top:6px}.preview{min-height:260px}.status{margin-top:14px;padding:10px}.top{margin-bottom:18px}.card{padding:20px}.grid{grid-template-columns:minmax(0,1fr) minmax(0,1fr)}@media(max-width:760px){.grid{grid-template-columns:1fr}.preview{min-height:180px}}\n</style></head><body><main class="wrap">\n<header class="top"><div><div class="eyebrow">MINIMAX H3</div><h1>製作有對白的影片</h1></div><span id="environment" class="pill pending">連線中…</span></header>\n<div class="grid"><div><section class="card">\n<div class="head"><h2><span class="step">01</span>選擇並生成</h2></div>\n<div class="field"><label for="preset">範例</label><div class="row"><select id="preset"><option value="live_action">實拍｜深夜的車站</option><option value="anime">動畫｜兩顆月亮</option><option value="animation_3d">3D 動畫｜小小太空人</option><option value="monochrome">黑白電影｜最後一次拍攝</option><option value="empty">自行輸入</option></select><button id="loadPreset">使用</button></div></div>\n<details id="promptEditor"><summary>編輯・複製提示詞</summary><div class="inside">\n<div class="field"><label for="prompt">提示詞</label><textarea class="prompt" id="prompt" spellcheck="false" maxlength="16000"></textarea><div class="count" id="charCount"></div></div>\n<div class="actions"><button id="copyPrompt">複製</button><button id="pastePrompt">貼上</button><button id="savePrompt">儲存為 .txt</button><button id="stripFence">移除程式碼框</button></div>\n</div></details>\n<details><summary>從圖片生成動態／Omni 參考圖</summary><div class="inside"><div class="cols"><div class="reference"><label for="firstImage">起始畫面</label><input id="firstImage" type="file" accept="image/png,image/jpeg,image/webp"><img id="firstPreview" alt="起始參考圖片" class="hidden"><p class="hint" id="firstLabel">無圖片</p><button class="small" id="clearFirst">移除</button></div><div class="reference"><label for="lastImage">結束畫面（選填）</label><input id="lastImage" type="file" accept="image/png,image/jpeg,image/webp"><img id="lastPreview" alt="結束參考圖片" class="hidden"><p class="hint" id="lastLabel">無圖片</p><button class="small" id="clearLast">移除</button></div></div><p class="inline-note">PNG・JPEG・WebP ／ 每個最大 10MB。Omni 模式：兩張圖依序是 &lt;Picture 1&gt;、&lt;Picture 2&gt;，請在提示詞裡用這些標籤引用（最多 9 張請用代理模式）</p></div></details>\n<div class="cols space"><div class="field"><label for="size">畫面</label><select id="size"><option value="landscape">橫向</option><option value="portrait">直向</option><option value="square">正方形</option><option value="native">橫向・高解析度</option><option value="p480">草稿 480p 橫向（832×480）</option><option value="p480v">草稿 480p 直向（480×832）</option></select></div><div class="field"><label for="seconds">長度</label><select id="seconds"><option value="5">約 5 秒</option><option value="8">約 8 秒</option><option value="10">約 10 秒</option><option value="15">約 15 秒</option></select></div></div>\n<details id="advanced"><summary>進階設定・儲存</summary><div class="inside"><div class="cols"><div class="field"><label for="mode">品質</label><select id="mode"><option value="turbo">快速 · Turbo 8 步</option><option value="turbo4">極速 · Turbo 4 步（768p 調校）</option><option value="omni">Omni · 參考圖生成（8 步，需下載 FastH3 Omni）</option><option value="omni_hd">Omni 草稿＋二次取樣 720p（尺寸請選草稿 480p）</option><option value="ref4">Ref2VA 4 步草稿（參考圖，需下載）</option><option value="quality" selected>標準 · 優先動態與聲音</option></select></div><div class="field"><label for="seed">Seed（-1 為隨機）</label><input id="seed" type="number" value="-1" min="-1" max="4294967295" step="1"></div></div><div class="field hidden" id="stepsField"><label for="steps">步數</label><select id="steps"><option value="16">16</option><option value="20" selected>20</option><option value="25">25</option><option value="30">30</option></select></div><p class="duration" id="duration"></p><div class="actions"><button id="importButton">載入設定</button><button id="exportConfig">儲存設定</button><input hidden class="hidden" id="configFile" type="file" accept="application/json,.json"></div></div></details>\n<button class="primary run" id="generate">生成影片</button><p class="estimate" id="estimate">預估 1〜5 分・首次較久</p><button class="run hidden" id="cancel">中止</button>\n<div id="status" role="status" aria-live="polite" class="status">確認準備中</div>\n<div id="runProgress" class="hidden"><div class="stages" aria-label="生成流程"><span data-stage="loading">載入</span><span data-stage="sampling">生成</span><span data-stage="saving">儲存</span></div><progress id="progressBar" max="1" aria-label="目前處理進度"></progress><div class="progressMeta"><span id="progressDetail"></span><span id="elapsed"></span></div></div>\n</section></div><div>\n<section class="card"><div class="head"><h2><span class="step">02</span>儲存影片</h2></div><div id="emptyPreview" class="preview"><div class="icon">▷</div><div>影片會顯示在這裡</div></div><video id="video" controls playsinline class="hidden"></video><div id="resultMeta" class="resultmeta"></div><div id="resultActions" class="actions hidden"><button id="downloadVideo" class="primary">儲存 MP4</button><button id="downloadBundle">連同設定儲存（ZIP）</button><button id="reuse">再次使用</button></div><details id="resultDetails" class="hidden"><summary>影片設定</summary><div id="resultTechnical" class="inside resultmeta"></div></details><details><summary>歷史紀錄 <span id="historyCount"></span></summary><div id="history" class="inside"><span class="hint">尚未生成</span></div></details></section>\n</div></div><footer class="foot">儲存後，在 Colab 選擇「執行階段 → 中斷連線並刪除執行階段」即可結束。</footer>\n</main><div id="toast" class="toast hidden" role="status"></div>\n<script>\n(()=>{\'use strict\';\nconst $=id=>document.getElementById(id);\nconst frames={\'5\':124,\'8\':192,\'10\':243,\'15\':362};\nconst examples={\n "live_action": "integrated_multimodal_description: [Shot 1] Photorealistic Japanese suspense film, a single continuous five-second shot. An original Japanese woman in her late twenties, wearing a dark coat, stands on a deserted railway platform at night after rain. Waist-up framing, her face and mouth clearly visible, wet pavement and distant blurred station lights behind her. The camera makes a very slow push-in. She first looks toward the empty tracks offscreen, then meets the camera with restrained urgency and says in a low, breathy but intelligible voice (S1), <d>[Japanese] 次の電車には、乗らないで。</d> Natural Japanese speech, synchronized lip movements, one speaker, a brief tense silence after the line. Cool fluorescent light, realistic skin, subtle 35mm film texture. No train enters the shot. No subtitles, readable signs, logos or on-screen text.\\noverall_soundscape: A quiet electrical hum and faint dripping water beneath clear foreground dialogue.\\nnon_diegetic_music: N/A",\n "anime": "integrated_multimodal_description: [Shot 1] Original hand-drawn 2D anime, a single continuous five-second shot. An adult woman in her twenties with a short navy bob and an oversized rust-orange jacket stands on a rooftop at blue hour. Waist-up three-quarter view on the left of the frame, with two distinct moons clearly visible in the open sky on the right: one small familiar moon and one enormous pale turquoise moon. Clean expressive linework, cel shading, richly painted violet clouds; keep the image fully 2D, not photorealistic. A gentle breeze moves her hair. She glances at the sky, looks toward someone just beside the camera, and with quiet wonder in a clear natural voice (S1) says, <d>[Japanese] ねえ、月って二つあったっけ？</d> Natural Japanese pronunciation, synchronized anime mouth animation, one speaker. Locked camera, small expressive eye movements, hold on her puzzled expression at the end. No subtitles, logos or on-screen text.\\noverall_soundscape: A light rooftop breeze and distant city ambience beneath the voice.\\nnon_diegetic_music: N/A",\n "animation_3d": "integrated_multimodal_description: [Shot 1] Original stylized 3D animated science-fiction short, one continuous five-second shot. A small adult astronaut with rounded proportions and an expressive adult face sits inside a compact, worn spacecraft cockpit. An orange spacesuit, transparent open visor, tiny fabric seams and soft tactile materials. Medium close-up with the mouth fully visible; a vivid blue Earth glows through a round porthole behind the astronaut. Warm amber cabin lighting contrasts with the deep blue planet. The astronaut gently touches the porthole frame, turns back toward the camera with a tired, relieved smile, and says in a soft warm voice (S1), <d>[Japanese] 帰ろう。まだ、灯りがついてる。</d> Natural Japanese pronunciation, clear foreground speech and synchronized mouth movements. One speaker, subtle facial acting, static camera. Original character design, cinematic 3D rendering, no subtitles, readable screens or logos.\\noverall_soundscape: Soft spacecraft ventilation and a faint cabin creak beneath the voice.\\nnon_diegetic_music: N/A",\n "monochrome": "integrated_multimodal_description: [Shot 1] An original black-and-white romantic film scene, one continuous five-second shot. An adult actress in her late twenties with a softly waved bob and a plain dark blouse stands in a quiet, nearly empty film studio. A medium close-up at eye level. A single large side light shapes her face, distant studio equipment falls into soft shadow, rich grayscale contrast and fine film grain. She holds eye contact with someone just beside the lens, gives an almost-smile, and in a low, intimate voice with restrained vulnerability (S1) says, <d>[Japanese] カットって言うまで、好きでいて。</d> Natural Japanese pronunciation, precise lip synchronization, one speaker. The camera remains still and lingers on her face for a short silent beat after the line. Strictly monochrome, no color, subtitles, logos or on-screen text.\\noverall_soundscape: Quiet studio room tone beneath clear, close dialogue.\\nnon_diegetic_music: N/A",\n "empty": ""\n};\nlet active=null,current=null,results=[],ready=false,pollTimer=null,lastUrl=null,environment={},elapsedBase=0,elapsedAt=0;\nconst references={first_frame:\'\',last_frame:\'\'};let uploads=0;\nfunction toast(text){$(\'toast\').textContent=text;$(\'toast\').classList.remove(\'hidden\');clearTimeout(toast.timer);toast.timer=setTimeout(()=>$(\'toast\').classList.add(\'hidden\'),3300)}\nfunction setStatus(text,state=\'\'){ $(\'status\').textContent=text;$(\'status\').className=\'status \'+state;}\nfunction resize(){try{google.colab.output.setIframeHeight(document.documentElement.scrollHeight,true)}catch{}}\nasync function rpc(name,...args){\n let response;\n if(window.H3_TEST_BRIDGE){response=await window.H3_TEST_BRIDGE(name,...args)}\n else if(window.google?.colab?.kernel){const r=await google.colab.kernel.invokeFunction(\'h3lab.\'+name,args,{});response=r.data?.[\'application/json\'];if(typeof response===\'string\')response=JSON.parse(response);}\n else throw new Error(\'請在 Colab 執行①準備環境的儲存格。\');\n if(!response)throw new Error(\'無法連線到畫面。請重新執行最新版的①準備環境的儲存格。\');\n if(!response.ok)throw new Error(response.error||\'未收到執行階段的回應，請確認連線。\');return response.value;\n}\nasync function copyText(text){if(!text.trim())return toast(\'請輸入要複製的文字。\');try{await navigator.clipboard.writeText(text)}catch{const t=document.createElement(\'textarea\');t.value=text;t.style.position=\'fixed\';t.style.opacity=\'0\';document.body.append(t);t.select();const ok=document.execCommand(\'copy\');t.remove();if(!ok){toast(\'無法複製，請選取文字後自行複製。\');return}}toast(\'已複製。\')}\nfunction downloadBlob(blob,name){const url=URL.createObjectURL(blob),a=document.createElement(\'a\');a.href=url;a.download=name;document.body.append(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000)}\nfunction saveText(text,name,type=\'text/plain\'){downloadBlob(new Blob([text],{type:type+\';charset=utf-8\'}),name)}\nfunction withoutFence(text){return text.trim().replace(/^```[^\\n]*\\n/,\'\').replace(/\\n?```\\s*$/,\'\').trim()}\nfunction aligned(text){let body=withoutFence(text).replace(/^(?:For the target video,|How the reference pictures align with the target video)[^\\n]*\\n\\s*/,\'\');let end=(frames[$(\'seconds\').value]/24).toFixed(2),prefix=\'\';\n const shots=[...body.matchAll(/\\[Shot (\\d+)\\]/g)].map(m=>Number(m[1]));const lastShot=Math.max(1,...shots);\n if(references.first_frame&&references.last_frame)prefix=`How the reference pictures align with the target video — Picture 1 (from Shot 1) aligns with the 0.00-second mark of the target video; Picture 2 (from Shot ${lastShot}) aligns with the ${end}-second mark of the target video.`;\n else if(references.first_frame)prefix=\'For the target video, at 0.00 seconds into the target video, <Picture 1> (from [Shot 1]) is fully referenced.\';\n else if(references.last_frame)prefix=`How the reference pictures align with the target video — <Picture 1> (from [Shot ${lastShot}]) aligns with the ${end}-second mark of the target video.`;\n return prefix?prefix+\'\\n\\n\'+body:body;\n}\n\nfunction config(){return {prompt:aligned($(\'prompt\').value),size:$(\'size\').value,seconds:$(\'seconds\').value,mode:$(\'mode\').value,steps:Number($(\'steps\').value),seed:Number($(\'seed\').value),...references}}\nfunction formatTime(seconds){const n=Math.max(0,Math.round(seconds));return n<60?n+\'秒\':Math.floor(n/60)+\'分\'+String(n%60).padStart(2,\'0\')+\'秒\'}\nfunction estimate(){const c=config(),match=results.find(r=>[\'size\',\'seconds\',\'mode\',\'steps\'].every(k=>String(r.config[k])===String(c[k]))&&r.environment?.profile===environment.profile&&r.environment?.hardware?.gpu===environment.hardware?.gpu);if(match)return \'上次 \'+formatTime(match.generation_seconds)+\' · 相同設定\';if(environment.profile===\'full\')return \'未量化：數分〜數十分（未測量）\';const factor=frames[c.seconds]/124*(c.mode===\'quality\'?c.steps/8:1)*(c.size===\'native\'?2:c.size===\'square\'?1.13:1);return `預估 ${Math.max(1,Math.round(factor))}〜${Math.ceil(5*factor)}分 · 首次較久`}\nfunction refresh(){ $(\'charCount\').textContent=$(\'prompt\').value.length.toLocaleString()+\'字\';$(\'stepsField\').classList.toggle(\'hidden\',$(\'mode\').value!==\'quality\');$(\'duration\').textContent=`實際長度 ${(frames[$(\'seconds\').value]/24).toFixed(2)}秒 / 24fps`;$(\'estimate\').textContent=estimate();$(\'generate\').disabled=!!active||!ready||uploads>0;resize();}\nfunction showProgress(s){$(\'runProgress\').classList.remove(\'hidden\');const names=[\'loading\',\'sampling\',\'saving\'],i=names.indexOf(s.stage);document.querySelectorAll(\'[data-stage]\').forEach((el,n)=>{el.classList.toggle(\'active\',s.state===\'running\'&&n===i);el.classList.toggle(\'done\',s.state===\'done\'||n<i)});const bar=$(\'progressBar\');if(s.state===\'done\'){bar.max=1;bar.value=1}else if(s.state===\'running\'&&s.total_steps>0){bar.max=s.total_steps;bar.value=s.step}else if(s.state===\'running\'){bar.removeAttribute(\'value\')}else{bar.max=1;bar.value=0}elapsedBase=s.elapsed||0;elapsedAt=Date.now();$(\'elapsed\').textContent=\'已過 \'+formatTime(elapsedBase);$(\'progressDetail\').textContent=s.state===\'running\'&&s.total_steps?`${s.step} / ${s.total_steps} 步`:s.state===\'running\'?\'處理中…\':s.state===\'done\'?\'完成\':\'已停止\';}\nsetInterval(()=>{if(active&&elapsedAt)$(\'elapsed\').textContent=\'已過 \'+formatTime(elapsedBase+(Date.now()-elapsedAt)/1000)},1000);\nfunction loadPreset(){const key=$(\'preset\').value;$(\'prompt\').value=examples[key];$(\'mode\').value=\'quality\';if(key===\'empty\'){$(\'promptEditor\').open=true;$(\'prompt\').focus()}refresh();} $(\'loadPreset\').onclick=loadPreset;$(\'preset\').onchange=loadPreset;\n$(\'copyPrompt\').onclick=()=>copyText(config().prompt);$(\'pastePrompt\').onclick=async()=>{try{$(\'prompt\').value=withoutFence(await navigator.clipboard.readText());refresh();toast(\'已貼上\')}catch{$(\'prompt\').focus();toast(\'請在此欄按 ⌘V / Ctrl+V 貼上\')}};$(\'savePrompt\').onclick=()=>saveText(config().prompt,\'minimax_h3_prompt.txt\');$(\'stripFence\').onclick=()=>{$(\'prompt\').value=withoutFence($(\'prompt\').value);refresh();toast(\'已移除程式碼框。\')};\n$(\'exportConfig\').onclick=()=>saveText(JSON.stringify({format:\'minimax-h3-colab-v1\',config:config()},null,2),\'minimax_h3_settings.json\',\'application/json\');\nfunction clearImage(key){references[key]=\'\';const first=key===\'first_frame\',p=first?\'first\':\'last\';$(p+\'Image\').value=\'\';$(p+\'Preview\').removeAttribute(\'src\');$(p+\'Preview\').classList.add(\'hidden\');$(p+\'Label\').textContent=\'無圖片\';refresh()}\nfunction restore(value,keepRefs=false){const c=value.config||value;if(typeof c.prompt!==\'string\'||![\'landscape\',\'portrait\',\'square\',\'native\'].includes(c.size)||!frames[String(c.seconds)]||![\'turbo\',\'turbo4\',\'omni\',\'omni_hd\',\'ref4\',\'quality\'].includes(c.mode))throw new Error(\'請選擇此筆記本的設定 JSON 檔。\');$(\'prompt\').value=c.prompt;for(const key of [\'size\',\'seconds\',\'mode\',\'seed\',\'steps\'])if(c[key]!==undefined)$(key).value=String(c[key]);if(!keepRefs){clearImage(\'first_frame\');clearImage(\'last_frame\');if(c.first_frame||c.last_frame)toast(\'已載入設定。參考圖片請重新選擇。\')}else{for(const key of [\'first_frame\',\'last_frame\']){clearImage(key);references[key]=c[key]||\'\';if(c[key])$(key===\'first_frame\'?\'firstLabel\':\'lastLabel\').textContent=\'重用此結果的參考圖片\'}}refresh()}\n$(\'importButton\').onclick=()=>$(\'configFile\').click();$(\'configFile\').onchange=async e=>{try{const f=e.target.files[0];if(!f)return;if(f.size>1000000)throw new Error(\'設定 JSON 請限制在 1MB 以內。\');restore(JSON.parse(await f.text()));toast(\'已載入設定。\')}catch(e){toast(e.message)}finally{$(\'configFile\').value=\'\'}};\nasync function upload(file,key){if(!file)return;const p=key===\'first_frame\'?\'first\':\'last\';if(file.size>10*1024*1024)return toast(\'圖片請限制在 10MB 以內。\');uploads++;refresh();$(p+\'Label\').textContent=\'上傳中…\';try{const data=await new Promise((resolve,reject)=>{const reader=new FileReader();reader.onload=()=>resolve(reader.result);reader.onerror=reject;reader.readAsDataURL(file)});const result=await rpc(\'upload\',data,file.name);references[key]=result.filename;$(p+\'Preview\').src=data;$(p+\'Preview\').classList.remove(\'hidden\');$(p+\'Label\').textContent=`${file.name} · ${result.width} × ${result.height}`;}catch(e){clearImage(key);toast(e.message)}finally{uploads--;refresh()}}\n$(\'firstImage\').onchange=e=>upload(e.target.files[0],\'first_frame\');$(\'lastImage\').onchange=e=>upload(e.target.files[0],\'last_frame\');$(\'clearFirst\').onclick=()=>clearImage(\'first_frame\');$(\'clearLast\').onclick=()=>clearImage(\'last_frame\');\nfunction busy(job){active=job;$(\'cancel\').classList.toggle(\'hidden\',!job);$(\'cancel\').disabled=false;refresh()}\nasync function poll(){if(!active)return;try{const s=await rpc(\'status\',active);setStatus(s.message,s.state===\'running\'?\'running\':s.state===\'done\'?\'success\':s.state===\'error\'?\'error\':\'\');showProgress(s);if(s.state===\'running\'){pollTimer=setTimeout(poll,2000);return}busy(null);if(s.state===\'done\'){results=[s.report,...results.filter(x=>x.id!==s.report.id)];renderHistory();await showResult(s.report);refresh();}}catch(e){setStatus(e.message+\'\\n重新執行①準備環境的儲存格即可重新連線。\',\'error\');showProgress({state:\'error\',elapsed:elapsedBase});busy(null)}}\n$(\'generate\').onclick=async()=>{if(!$(\'prompt\').value.trim())return setStatus(\'請輸入提示詞。\',\'error\');if(!Number.isInteger(Number($(\'seed\').value))||Number($(\'seed\').value)<-1||Number($(\'seed\').value)>4294967295)return setStatus(\'Seed 請輸入 -1 或 0〜4294967295 的整數。\',\'error\');$(\'generate\').disabled=true;try{const c=config();$(\'prompt\').value=c.prompt;const r=await rpc(\'run\',c);busy(r.id);setStatus(\'已開始生成\',\'running\');showProgress({state:\'running\',stage:\'loading\',elapsed:0});poll()}catch(e){setStatus(e.message,\'error\');refresh()}};\n$(\'cancel\').onclick=async()=>{if(!active)return;$(\'cancel\').disabled=true;try{const r=await rpc(\'cancel\',active);setStatus(r.message,\'running\')}catch(e){setStatus(e.message,\'error\');$(\'cancel\').disabled=false}};\nfunction toBlob(d){const bytes=Uint8Array.from(atob(d.base64),c=>c.charCodeAt(0));return new Blob([bytes],{type:d.mime})}\nasync function showResult(report){current=report;$(\'resultMeta\').textContent=`${report.config.actual_seconds.toFixed(1)}秒的影片 · 生成 ${formatTime(report.generation_seconds)}`;$(\'resultTechnical\').textContent=`${report.config.width} × ${report.config.height} · Seed ${report.config.seed} · ${report.environment.profile_label}`;$(\'resultDetails\').classList.remove(\'hidden\');$(\'resultActions\').classList.remove(\'hidden\');try{const d=await rpc(\'download\',report.id,\'video\');if(lastUrl)URL.revokeObjectURL(lastUrl);lastUrl=URL.createObjectURL(toBlob(d));$(\'video\').src=lastUrl;$(\'video\').classList.remove(\'hidden\');$(\'emptyPreview\').classList.add(\'hidden\')}catch(e){toast(e.message)}resize()}\nasync function saveResult(kind){if(!current)return;try{toast(\'正在準備要儲存的檔案…\');const d=await rpc(\'download\',current.id,kind);downloadBlob(toBlob(d),d.filename);toast(\'已開始下載。\')}catch(e){toast(e.message)}}\n$(\'downloadVideo\').onclick=()=>saveResult(\'video\');$(\'downloadBundle\').onclick=()=>saveResult(\'zip\');$(\'reuse\').onclick=()=>{if(current){restore(current,true);toast(\'已載入生成時的 Seed 與提示詞。\')}};\nfunction renderHistory(){$(\'historyCount\').textContent=results.length?`(${results.length})`:\'\';const root=$(\'history\');root.replaceChildren();if(!results.length){root.textContent=\'尚未生成。\';return}for(const r of results){const item=document.createElement(\'div\');item.className=\'historyitem\';const p=document.createElement(\'p\');p.textContent=r.config.prompt.replace(/^integrated_multimodal_description:\\s*/,\'\');p.title=r.config.prompt;item.append(p);const line=document.createElement(\'div\');line.className=\'resultmeta\';line.textContent=`${r.created_at} · ${r.config.actual_seconds.toFixed(2)}秒 · Seed ${r.config.seed}`;item.append(line);const actions=document.createElement(\'div\');actions.className=\'actions\';const open=document.createElement(\'button\');open.textContent=\'檢視\';open.onclick=()=>showResult(r);const reuse=document.createElement(\'button\');reuse.textContent=\'重用設定\';reuse.onclick=()=>{restore(r,true);toast(\'已重用設定。\')};actions.append(open,reuse);item.append(actions);root.append(item)}resize()}\nfor(const id of [\'prompt\',\'size\',\'seconds\',\'mode\',\'steps\',\'seed\'])$(id).addEventListener(\'input\',refresh);document.querySelectorAll(\'details\').forEach(d=>d.addEventListener(\'toggle\',resize));\nasync function init(){ $(\'prompt\').value=examples.live_action;refresh();try{const a=await rpc(\'about\');ready=a.ready;environment=a.environment||{};results=a.results||[];renderHistory();$(\'environment\').textContent=ready?(a.environment.hardware?.gpu||\'GPU\')+\' · 準備完成\':\'需要設定\';$(\'environment\').classList.toggle(\'pending\',!ready);setStatus(ready?\'可直接按「生成影片」\':\'請執行①準備環境的儲存格\');if(a.active){busy(a.active);poll()}}catch(e){setStatus(e.message,\'error\');$(\'environment\').textContent=\'無法連線\'}refresh()}\nwindow.H3_UI_TEST={config,restore,aligned,examples};init();\n})();\n</script></body></html>\n'
from IPython.display import HTML, display
profile = 'full' if model_precision.startswith('未量化') else ('int8enc' if model_precision.startswith('INT8＋') else 'int8')
try:
    setup_report = h3.setup(profile, download_omni, download_ref4)
    if setup_report:
        h3.install_callbacks()
        display(HTML(UI_SOURCE))
except Exception as error:
    import html
    display(HTML('<div style="padding:14px;color:#922c36">' + html.escape(str(error)) + '</div>'))


In [ ]:
#@title ② 代理模式：接收 agent 從 Google Drive 送來的工作（執行期間會持續占用此 cell）
#@markdown 先跑過 ①。這個 cell 會一直執行並輪詢 Drive，期間 ① 的操作畫面按鈕會沒有反應（Colab 一次只跑一個 cell）。要停止：按 ■ 中斷（執行中的工作會退回 inbox，下次自動重跑），或在 h3-queue 資料夾建立名為 STOP 的檔案（每次啟動時會自動清掉舊的 STOP）。
queue_folder = "h3-queue" #@param {type:"string"}
poll_seconds = 30 #@param {type:"integer"}
import sys
h3 = sys.modules.get('minimax_h3_colab_backend_v3')
if h3 is None or not h3.BASE.joinpath('setup.json').exists():
    raise RuntimeError('請先執行 ① 完成環境準備。')
from google.colab import drive
drive.mount('/content/drive')
root = h3.queue_init('/content/drive/MyDrive/' + queue_folder)
print('佇列資料夾：', root)
print('代理協定說明在下方的說明儲存格。')
h3.agent_loop(root, poll=poll_seconds)


## 說明

- 手動模式用 ①；agent 模式用 ②，兩者同一時間只能跑一個（Colab 單 cell 限制）。
- 模式：`omni_hd`（Omni 草稿 480p＋同一次工作內的二次取樣，輸出 1280×720；草稿尺寸用 `p480`／`p480v`，`refine_steps` 預設 4）、`ref4`（Ref2VA 4 步草稿，參考圖專用）、`turbo`（8 步）、`turbo4`（4 步，最快，768p 調校）、`omni`（FastH3 Omni，8 步，用 1～9 張參考圖，提示詞以 `<Picture 1>` 引用）、`quality`（20 步起）。同時只套用一個加速 LoRA，不疊加；Omni 本身已蒸餾，不再疊 LoRA。
- Omni 目前只支援圖片參考，影片／音訊參考尚未接上；Omni 兩種 sampler（預設 `res_multistep+simple`、`euler+beta`）都實測過：角色特徵都保留、暖機後速度相同（5 秒影片約 55 秒），用預設即可；首次載入 Omni 模型會多出約 2 分鐘。
- Colab 工作階段最長 12 小時（Pro+ 單位足夠時可到 24 小時），斷線後重跑 ① ② 即可；中斷時卡在 running 的工作會自動退回 inbox 重跑。
- 結果也在 `/content/h3-lab/results`，刪除執行階段就會消失。
- 模型授權：MiniMax H3 社群授權，使用前請自行確認你所在地區是否可使用。

[模型](https://huggingface.co/MiniMaxAI/MiniMax-H3) · [ComfyUI 用模型](https://huggingface.co/Comfy-Org/MiniMax-H3) · [Colab FAQ](https://research.google.com/colaboratory/faq.html)


## Agent 送件協定

筆記本不開公網。agent 與 Colab 之間只透過 Google Drive 的 `h3-queue` 資料夾交換檔案。
Colab 端的「代理模式」cell 必須正在執行，agent 才拿得到結果。

## 資料夾（在 My Drive 根目錄，筆記本啟動時自動建立）

```
h3-queue/
  inbox/    agent 把工作單放這裡
  running/  筆記本認領後移到這裡（agent 不要碰）
  done/     <id>.json（完成標記）+ <id>.mp4 + <id>.settings.json
  failed/   <id>.json（含 error 訊息）
  heartbeat.json   筆記本每輪更新；每約 5 分鐘更新；alive_at 超過約 10 分鐘沒動 = 筆記本已停
  STOP      （選用）建立此檔可讓筆記本的代理迴圈結束
```

## 怎麼存取這個資料夾（優先順序）

1. **本機有 Google Drive 桌面版同步（這台電腦是 `G:\我的雲端硬碟`）：直接當一般檔案讀寫**，
   例如寫入 `G:\我的雲端硬碟\h3-queue\inbox\`、讀取 `...\done\`。mp4 幾 MB 到幾十 MB，這條路最穩。
2. 遠端 agent 才用 Google Drive MCP：只拿來丟小的 JSON / 圖片、查完成標記。**不要用 `download_file_content` 抓 mp4**
   （會把 base64 整包塞進對話，放不下）；改成回報檔案 ID 或連結，讓使用者或本機端取。

兩個方向的同步都是非同步的，檔案可能「先看到標記、後看到內容」，所以一律用下面的規則防呆。

## 送件（順序很重要）

1. 選一個 id：只能用英數、`-`、`_`，最長 64 字，必須唯一（例如 `clip-20261022-001`）。
2. 若有參考圖，**先**放到 `inbox/`：`<id>.first_frame.png`、`<id>.last_frame.png`（選用），
   並在工作單裡用 `"images": ["first_frame"]` 宣告。筆記本會等宣告的圖片都到齊才開始（最多等 10 分鐘，逾時算失敗），
   不會悄悄變成純文字生影片。
3. **最後**才放 `inbox/<id>.json`（筆記本只掃 .json）。

用 Drive MCP 的 `create_file` 時：`parentId` = inbox 資料夾 ID、`title` = `<id>.json`、
`contentMimeType` = `application/json`，**必須設 `disableConversionToGoogleType: true`**，否則會被轉成 Google 文件。
圖片用 `base64Content` + `image/png`。

## 工作單欄位

```json
{
  "prompt": "integrated_multimodal_description: ...\noverall_soundscape: ...\nnon_diegetic_music: N/A",
  "size": "landscape",
  "seconds": "5",
  "mode": "turbo",
  "seed": -1,
  "images": ["first_frame"]
}
```

| 欄位 | 值 |
|---|---|
| prompt | 必填，16000 字內 |
| size | `p480`(832x480，草稿) `p480v`(480x832) `landscape`(960x544) `portrait`(544x960) `square`(768x768) `native`(1344x768) |
| seconds | `"5"` `"8"` `"10"` `"15"` |
| mode | `omni_hd`（Omni 草稿加二次取樣，輸出 1280×720，至少 1 張參考圖；`size` 只能 `p480` 或 `p480v`，預設 `p480`；可選 `refine_steps` 2～6，預設 4）`ref4`（Ref2VA 4 步，參考圖專用）`turbo`（8 步，預設）`turbo4`（4 步，最快；LoRA 為 768p 調校，建議配 `native` 或 `square` 比較）`omni`（FastH3 Omni，8 步，參考圖生成，**至少 1 張參考圖**）`quality`（20 步起，品質優先）`ref2va`（官方非蒸餾 Ref2VA，單階段，參考圖專用）`ref2va_hd`（Ref2VA 草稿加尾段二次取樣，輸出 1280×720，參考圖專用）`ayase`（Ayase 式兩段：Ref2VA 剪枝 INT8＋HyperFlow 8 步 LoRA、SigmaShift 12/3，960×544 草稿 8 步，潛空間放大後再跑二採；預設畫面 `landscape`，`upscale` 2.0 時輸出 1920×1088；需要 download_ref4 勾選，參考圖專用；實測 5 秒可跑、15 秒在 A100 40 GB 失敗，建議先用 5 秒） |
| steps | 只有 `quality` 有效：16 / 20 / 25 / 30。`ref2va` 與 `ref2va_hd`：12～30，預設 20 |
| refine_steps | 只有 `ref2va_hd` 用：2～12，預設 8，必須小於 `steps`（尾段取樣的步數）。`omni_hd` 的設定見上表 mode 列 |
| upscale | 只有 `ayase` 用：1.25～2.5，預設 2.0（潛空間放大倍率） |
| second_steps | 只有 `ayase` 用：3～12，預設 7（第二段取樣步數） |
| second_denoise | 只有 `ayase` 用：0.05～0.6，預設 0.22（第二段去噪強度） |
| seed | -1 隨機，或 0–4294967295 |
| images | 宣告隨附的圖片：一般模式 `first_frame`、`last_frame`；`omni` 模式 `ref_1`…`ref_9`（依序對應提示詞裡的 `<Picture 1>`…`<Picture 9>`） |
| sampler | 選用：`res_multistep+simple`（預設）或 `euler+beta`。兩者實測畫質相近、速度相同，用預設即可 |

其他欄位一律忽略；LoRA 不能自選。

## 取件

1. 輪詢 `done/<id>.json`（每 30–60 秒一次）。出現後讀 `video_bytes`，
   **等 `done/<id>.mp4` 的檔案大小等於 `video_bytes` 才算完成**（同步可能先送標記）。
2. 同時輪詢 `failed/<id>.json`；有的話讀 `error` 欄位，修正後換新 id 重送。
3. 逾時判斷：`heartbeat.json` 約每 5 分鐘更新一次；`alive_at` 超過約 10 分鐘沒動，就表示筆記本已停。
4. 筆記本中斷（■ 或 Colab 斷線）時，執行中的工作會退回 `inbox/` 等下次啟動重跑，不需要重送。

## Omni 模式（FastH3 Omni 參考圖生成）

- 圖片檔 `<id>.ref_1.png`…`<id>.ref_9.png`，工作單 `"mode": "omni"`、`"images": ["ref_1","ref_2"]`。
- 提示詞用 `<Picture 1>`、`<Picture 2>` 引用對應的圖；`ref_N` 對應 `<Picture N>`，編號必須從 1 開始連續（跳號會被拒絕）。
- 目前只支援圖片參考；影片／音訊參考尚未接上。
- 模型已蒸餾成 8 步，不要再疊 turbo LoRA（筆記本也不會疊）。
- 需要 ① 勾選「download_omni」下載模型（INT8 約 +22GB），否則工作單會進 `failed/`。

## 草稿再精修（omni_hd）與完成後的檔案

- 一個工作內完成：Omni 8 步草稿（480p）→ 解碼、放大到 1280×736 → 用同一個模型只跑排程尾段的二次取樣 → 裁成 1280×720。
- 想「抽卡」：先用 `omni` 搭配 `size: p480` 與不同 seed 快速出草稿（`seconds` 可 5／10／15），挑到喜歡的 seed 後，用同一個 seed、同一份提示詞與參考圖改 `mode: omni_hd` 重跑；同 seed 的草稿是否完全重現，以實測為準（見測試結果）。
- `done/<id>.settings.json` 內有 `timeline`（每個節點開始的秒數）與 `min_vram_free_gib`（工作期間最低剩餘顯存），`omni_hd` 另有 `done/<id>.draft.mp4`（無聲的草稿，用來和精修後比對）。
